# Put bermudéen — Longstaff-Schwartz

On implémente l’algorithme de Longstaff-Schwartz pour un put bermudéen dans Black-Scholes.

Paramètres :
\[
r=0.1,\ \sigma=0.25,\ x_0=100,\ K=110,\ N=10,\ T=1.
\]

Dates d’exercice : \(t_k=k/N\), \(k=0,\dots,N\).  
Payoff :
\[
\phi_k(x)=e^{-rk/N}(K-x)_+.
\]

On utilise la récurrence :
\[
V_N(x)=\phi_N(x),\qquad
V_k(x)=\max\big(\phi_k(x),\,C_k(x)\big),\ k=N-1,\dots,0,
\]
avec
\[
C_k(x)=\mathbb E\big[V_{k+1}(X_{k+1})\mid X_k=x\big].
\]

Dans LS, \(C_k\) est approchée par régression sur des trajectoires Monte Carlo (sur les trajectoires ITM).

On affichera ensuite les fonctions apprises \(\widehat C_k\) et
\[
\widehat V_k(x)=\max\big(\phi_k(x),\widehat C_k(x)\big).
\]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def simulate_bs_paths(x0, r, sigma, T, N, n_paths, seed=123):
    dt = T / N
    rng = np.random.default_rng(seed)

    z = rng.standard_normal((n_paths, N))
    X = np.empty((n_paths, N + 1), dtype=float)
    X[:, 0] = x0

    drift = (r - 0.5 * sigma**2) * dt
    vol = sigma * np.sqrt(dt)

    for k in range(N):
        X[:, k + 1] = X[:, k] * np.exp(drift + vol * z[:, k])

    return X


def payoff_discounted(x, k, K, r, dt):
    return np.exp(-r * k * dt) * np.maximum(K - x, 0.0)


def basis_poly2(x):
    x = np.asarray(x)
    return np.column_stack([np.ones_like(x), x, x**2])


def fit_continuation_model(x_itm, y_itm):
    if x_itm.size == 0:
        return {"kind": "const", "c": 0.0}

    if np.unique(np.round(x_itm, 10)).size < 3:
        return {"kind": "const", "c": float(np.mean(y_itm))}

    A = basis_poly2(x_itm)
    beta, *_ = np.linalg.lstsq(A, y_itm, rcond=None)
    return {"kind": "poly2", "beta": beta}


def predict_continuation(model, x):
    x = np.asarray(x)

    if model["kind"] == "poly2":
        y = basis_poly2(x) @ model["beta"]
    else:
        y = np.full_like(x, model["c"], dtype=float)

    return np.maximum(y, 0.0)

In [ ]:
def longstaff_schwartz_bermudan_put(x0, K, r, sigma, T, N, n_paths=40000, seed=123):
    dt = T / N
    X = simulate_bs_paths(x0, r, sigma, T, N, n_paths, seed=seed)

    cashflow = payoff_discounted(X[:, N], N, K, r, dt)
    tau = np.full(n_paths, N, dtype=int)

    models = [None] * (N + 1)
    models[N] = {"kind": "terminal"}

    for k in range(N - 1, -1, -1):
        immediate = payoff_discounted(X[:, k], k, K, r, dt)
        itm = immediate > 0.0

        y_itm = cashflow[itm]
        x_itm = X[itm, k]

        model_k = fit_continuation_model(x_itm, y_itm)
        models[k] = model_k

        cont_all = predict_continuation(model_k, X[:, k])
        exercise_now = itm & (immediate >= cont_all)

        cashflow = np.where(exercise_now, immediate, cashflow)
        tau = np.where(exercise_now, k, tau)

    price = float(np.mean(cashflow))

    return {
        "price": price,
        "models": models,
        "X": X,
        "tau": tau,
        "cashflow": cashflow,
        "dt": dt,
        "params": {
            "x0": x0,
            "K": K,
            "r": r,
            "sigma": sigma,
            "T": T,
            "N": N,
            "n_paths": n_paths,
            "seed": seed,
        },
    }

In [ ]:
r = 0.1
sigma = 0.25
x0 = 100
K = 110
N = 10
T = 1.0

n_paths = 40000
seed = 7

res = longstaff_schwartz_bermudan_put(
    x0=x0,
    K=K,
    r=r,
    sigma=sigma,
    T=T,
    N=N,
    n_paths=n_paths,
    seed=seed,
)

print(f"Prix estimé (LS) : {res['price']:.6f}")

In [ ]:
def continuation_on_grid(models, k, x_grid):
    if k == len(models) - 1:
        return np.zeros_like(x_grid)
    return predict_continuation(models[k], x_grid)


def value_on_grid(models, k, x_grid, K, r, dt):
    phi = payoff_discounted(x_grid, k, K, r, dt)
    if k == len(models) - 1:
        return phi
    C = continuation_on_grid(models, k, x_grid)
    return np.maximum(phi, C)


x_grid = np.linspace(40, 180, 400)
models = res["models"]
K = res["params"]["K"]
r = res["params"]["r"]
dt = res["dt"]
N = res["params"]["N"]

fig, axes = plt.subplots(2, 5, figsize=(18, 6), sharex=True, sharey=True)
axes = axes.ravel()

for k in range(N):
    Ck = continuation_on_grid(models, k, x_grid)
    axes[k].plot(x_grid, Ck, color="tab:blue", lw=2, label=fr"$\hat C_{k}(x)$")
    axes[k].set_title(f"k={k}")
    axes[k].grid(alpha=0.3)

axes[0].legend(loc="upper right", fontsize=9)
fig.suptitle("Fonctions de continuation $\\hat C_k$", fontsize=14)
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(18, 10), sharex=True, sharey=True)
axes = axes.ravel()

for k in range(N + 1):
    Vk = value_on_grid(models, k, x_grid, K, r, dt)
    phik = payoff_discounted(x_grid, k, K, r, dt)

    axes[k].plot(x_grid, Vk, color="tab:green", lw=2, label=fr"$\hat V_{k}(x)$")
    axes[k].plot(x_grid, phik, color="tab:red", lw=1.5, ls="--", label=fr"$\phi_{k}(x)$")
    axes[k].set_title(f"k={k}")
    axes[k].grid(alpha=0.3)

axes[-1].axis("off")
axes[0].legend(loc="upper right", fontsize=8)
fig.suptitle("Fonctions valeurs $\\hat V_k$", fontsize=14)
fig.tight_layout()
plt.show()